# Modul B · Kapitel 2.3 — Semantic Search

## Challenge: Nach Bedeutung suchen, nicht nach Wörtern


**Lernziel:** Du kannst Texte einbetten, Ähnlichkeit messen und eine semantische Suche über eine
Vector Database bauen.

Dieses Notebook baut das Retrieval der RAG-Kette:

```
Dokumente ──► Chunks ──► Embeddings ──► Vector Database ──► Retrieval ──► Prompt ──► Antwort
                            └──────────────── hier ───────────────┘
```

Ein Embedding-Modell bildet jeden Chunk auf einen Vektor ab. Die Frage geht durch dasselbe
Modell und wird ebenfalls zu einem Vektor. Gesucht wird dann über den Abstand zwischen den
Vektoren: Die Treffer sind die nächsten Nachbarn der Frage.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die
Funktionen, die du vorher schreibst.

Es sind insgesamt **5 Challenges**.

---
## 0 · Setup

▶️ Führe die beiden nächsten Zellen aus. Die erste holt die Pakete und stellt die Diagramme ein.

In [ ]:
# ▶️ Pakete und Diagramm-Einstellungen
import json
import math
import re
import sys
import time
from pathlib import Path

try:
    import chromadb
    import numpy as np
    from rank_bm25 import BM25Okapi
except ImportError:
    %pip install -q chromadb numpy rank-bm25
    import chromadb
    import numpy as np
    from rank_bm25 import BM25Okapi

import matplotlib.pyplot as plt

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

print(f"chromadb {chromadb.__version__}, numpy {np.__version__}")
print("Setup fertig ✔")

▶️ Die zweite Zelle lädt `helfer.py`, holt die vorbereiteten Chunks und legt die Vector Database
an.

In `helfer.py` steht auch die Verbindung zum Modell — an genau einer Stelle, damit ein
Modellwechsel nur eine Datei betrifft:

```python
from openai import OpenAI

# Ollama, lokal. Für einen Provider: base_url und api_key tauschen, Modellname anpassen.
BASIS_URL = "http://localhost:11434/v1"
API_KEY = "ollama"                # Ollama prüft den Key nicht
MODELL = "qwen3.5:0.8b"           # das Chat-Modell
EMBEDDING_MODELL = "nomic-embed-text"

client = OpenAI(base_url=BASIS_URL, api_key=API_KEY)
```

Dieses Notebook stellt keine Frage an ein Chat-Modell. Es braucht nur das Embedding-Modell
`nomic-embed-text` — und später `bge-m3` zum Vergleich. Beide laufen lokal über **Ollama**. Für
einen anderen Provider tauschst du `BASIS_URL`, `API_KEY` und den Namen des Embedding-Modells,
und zwar in `helfer.py`, nicht hier.

In Google Colab gibt es kein lokales Ollama. Dort brauchst du einen Provider, der Embeddings
anbietet.

`helfer.embed()` merkt sich jeden berechneten Vektor in `daten/embedding_cache.json` — deshalb
ist die Collection in wenigen Sekunden gebaut.

In [ ]:
# ▶️ helfer.py finden, Chunks laden, Vector Database bauen
for kandidat in [Path.cwd(), *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

import helfer
from helfer import client, EMBEDDING_MODELL

chunks = helfer.lade_chunks()
fragen = helfer.lade_fragen()
sammlung = helfer.baue_chroma(chunks, neu=True)

print(f"Embedding-Modell: {EMBEDDING_MODELL}")
print(f"{len(chunks)} Chunks, {len(fragen)} Evaluationsfragen")
print(f"Collection {sammlung.name!r}: {sammlung.count()} Einträge unter {helfer.CHROMA_PFAD}")
print()
print(f"Erster Chunk: [{chunks[0]['chunk_id']}] {chunks[0]['titel'][:60]}")

---
## 1 · Ein Text wird zu einem Vektor

📖 Ein **Embedding-Modell** liest einen Text und gibt eine feste Zahl von Fließkommazahlen
zurück. Bei `nomic-embed-text` sind es 768. Diese Zahlen sind keine Wörter und keine Zählungen —
sie sind die Koordinaten des Textes in einem Raum mit 768 Achsen.

Der Raum ist so trainiert, dass Texte mit ähnlicher Bedeutung nahe beieinander liegen. Damit
wird die Suche zu einer geometrischen Aufgabe: Frage einbetten, nächste Nachbarn nehmen.

▶️ Die nächste Zelle bettet einen Satz ein und sieht sich den Vektor an.

In [ ]:
# ▶️ Ein Text, ein Vektor
BEISPIEL = "Ein Angreifer kann die Anmeldung umgehen."

vektor = helfer.embed([BEISPIEL])[0]
laenge = math.sqrt(sum(x * x for x in vektor))

print(f"Text:          {BEISPIEL!r}")
print(f"Modell:        {EMBEDDING_MODELL}")
print(f"Typ:           {type(vektor).__name__} aus {type(vektor[0]).__name__}")
print(f"Dimension:     {len(vektor)}")
print(f"Wertebereich:  {min(vektor):+.4f} bis {max(vektor):+.4f}")
print(f"Länge |v|:     {laenge:.4f}")
print()
print("Die ersten acht Zahlen:")
print("  " + "  ".join(f"{x:+.4f}" for x in vektor[:8]))

📖 Drei Beobachtungen.

**Es ist ein gewöhnlicher Vektor.** Eine Python-Liste aus 768 `float`. Nichts daran ist
besonders, außer wie er zustande kommt.

**Die Zahlen sind klein und wechseln das Vorzeichen.** Einzelne Werte bedeuten nichts. Es gibt
keine Achse „Schweregrad" und keine Achse „Runbook". Bedeutung steckt nur im Verhältnis der
Vektoren zueinander.

**Die Länge ist 1.** Dieses Modell gibt normalisierte Vektoren zurück — alle liegen auf einer
Kugel um den Ursprung. Das ist nicht selbstverständlich, und es wird im übernächsten Abschnitt
wichtig.

Ein Chunk und eine Frage gehen durch dasselbe Modell und landen im selben Raum. Nur deshalb
lassen sie sich überhaupt vergleichen.

---
## 2 · Drei Maße für Ähnlichkeit

📖 „Nah beieinander" muss man ausrechnen können. Drei Maße stehen zur Wahl:

| Maß | Formel | Wertebereich | Was es misst |
|---|---|---|---|
| **euklidische Distanz** | `sqrt(Σ (aᵢ - bᵢ)²)` | 0 … ∞ | den Abstand der beiden Punkte. Klein ist ähnlich. |
| **Skalarprodukt** | `Σ aᵢ · bᵢ` | -∞ … ∞ | Richtung **und** Länge. Groß ist ähnlich. |
| **Kosinus-Ähnlichkeit** | `Σ aᵢ · bᵢ / (‖a‖ · ‖b‖)` | -1 … 1 | den Winkel zwischen den Vektoren, ohne die Länge. Groß ist ähnlich. |

Dabei ist `‖a‖` die Länge des Vektors `a`, also `sqrt(Σ aᵢ²)`. Die Kosinus-Ähnlichkeit ist das
Skalarprodukt geteilt durch die beiden Längen — und damit dasselbe wie das Skalarprodukt der auf
Länge 1 gebrachten Vektoren.

Drei Werte kann man sich merken: Zwei gleiche Richtungen ergeben Kosinus 1, zwei rechtwinklige
ergeben 0, zwei entgegengesetzte ergeben -1.

### 🛠️ Challenge 1: Die drei Maße selbst rechnen

Schreibe die drei Funktionen. Jede nimmt zwei gleich lange Zahlenfolgen und gibt eine Zahl
zurück — **ohne `numpy` und ohne fertige Distanzfunktion**, nur mit Python.

| Funktion | Rückgabe |
|---|---|
| `skalarprodukt(a, b)` | die Summe der paarweisen Produkte |
| `euklidische_distanz(a, b)` | die Wurzel aus der Summe der quadrierten Differenzen |
| `kosinus_aehnlichkeit(a, b)` | das Skalarprodukt, geteilt durch das Produkt der beiden Längen |

Die Länge eines Vektors ist `math.sqrt(skalarprodukt(a, a))` — `skalarprodukt` lässt sich also
in `kosinus_aehnlichkeit` wiederverwenden. Ein Vektor der Länge 0 hat keine Richtung; gib in dem
Fall `0.0` zurück, statt durch null zu teilen.

*Tipp: `zip(a, b)` läuft über beide Folgen gleichzeitig, `math.sqrt(x)` zieht die Wurzel.*

In [ ]:
def skalarprodukt(a, b):
    """Summe der paarweisen Produkte zweier gleich langer Vektoren."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 1: skalarprodukt() implementieren")


def euklidische_distanz(a, b):
    """Abstand zweier Vektoren: Wurzel aus der Summe der quadrierten Differenzen."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 1: euklidische_distanz() implementieren")


def kosinus_aehnlichkeit(a, b):
    """Skalarprodukt geteilt durch die beiden Vektorlängen — der Kosinus des Winkels."""
    # TODO 1: die Länge von a und die Länge von b ausrechnen
    # TODO 2: das Skalarprodukt durch das Produkt der Längen teilen, Länge 0 abfangen
    raise NotImplementedError("Challenge 1: kosinus_aehnlichkeit() implementieren")


In [ ]:
# ✅ Selbsttest
OSTEN, NORDEN, WESTEN, WEIT_OSTEN = [1, 0, 0], [0, 1, 0], [-1, 0, 0], [3, 0, 0]

# bekannte Werte
assert abs(euklidische_distanz(OSTEN, OSTEN)) < 1e-12, "Ein Vektor hat zu sich selbst Abstand 0"
assert abs(euklidische_distanz(OSTEN, NORDEN) - math.sqrt(2)) < 1e-12, "Rechtwinklig: Abstand sqrt(2)"
assert abs(skalarprodukt(OSTEN, NORDEN)) < 1e-12, "Rechtwinklig: Skalarprodukt 0"
assert abs(kosinus_aehnlichkeit(OSTEN, OSTEN) - 1) < 1e-12, "Gleiche Richtung: Kosinus 1"
assert abs(kosinus_aehnlichkeit(OSTEN, NORDEN)) < 1e-12, "Rechtwinklig: Kosinus 0"
assert abs(kosinus_aehnlichkeit(OSTEN, WESTEN) + 1) < 1e-12, "Gegenläufig: Kosinus -1"
assert kosinus_aehnlichkeit([0, 0, 0], OSTEN) == 0.0, "Der Nullvektor darf nicht durch null teilen"

# der Unterschied zwischen den Maßen: gleiche Richtung, dreifache Länge
assert abs(kosinus_aehnlichkeit(OSTEN, WEIT_OSTEN) - 1) < 1e-12, "Kosinus ignoriert die Länge"
assert abs(skalarprodukt(OSTEN, WEIT_OSTEN) - 3) < 1e-12, "Das Skalarprodukt wächst mit der Länge"
assert abs(euklidische_distanz(OSTEN, WEIT_OSTEN) - 2) < 1e-12, "Die Distanz wächst mit der Länge"

# gegen numpy als Referenz
rng = np.random.default_rng(7)
x = rng.normal(size=64).tolist()
y = rng.normal(size=64).tolist()
assert abs(skalarprodukt(x, y) - float(np.dot(x, y))) < 1e-9, "Weicht von numpy ab"
assert abs(euklidische_distanz(x, y) - float(np.linalg.norm(np.subtract(x, y)))) < 1e-9, \
    "Weicht von numpy ab"
assert abs(kosinus_aehnlichkeit(x, y)
           - float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))) < 1e-9, \
    "Weicht von numpy ab"

print("✅ Challenge 1 gelöst")
print(f"{'Fall':<30}{'euklidisch':>12}{'Skalarprod.':>13}{'Kosinus':>10}")
print("-" * 65)
for name, a, b in [("gleiche Richtung", OSTEN, OSTEN),
                   ("rechtwinklig", OSTEN, NORDEN),
                   ("gegenläufig", OSTEN, WESTEN),
                   ("gleiche Richtung, 3-fach", OSTEN, WEIT_OSTEN)]:
    print(f"{name:<30}{euklidische_distanz(a, b):>12.4f}"
          f"{skalarprodukt(a, b):>13.4f}{kosinus_aehnlichkeit(a, b):>10.4f}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def skalarprodukt(a, b):
    """Summe der paarweisen Produkte zweier gleich langer Vektoren."""
    return sum(x * y for x, y in zip(a, b))


def euklidische_distanz(a, b):
    """Abstand zweier Vektoren: Wurzel aus der Summe der quadrierten Differenzen."""
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))


def kosinus_aehnlichkeit(a, b):
    """Skalarprodukt geteilt durch die beiden Vektorlängen — der Kosinus des Winkels."""
    laenge_a = math.sqrt(skalarprodukt(a, a))
    laenge_b = math.sqrt(skalarprodukt(b, b))
    if laenge_a == 0 or laenge_b == 0:
        return 0.0
    return skalarprodukt(a, b) / (laenge_a * laenge_b)
```

`zip(a, b)` bricht bei der kürzeren der beiden Folgen ab. Das ist bequem und gefährlich
zugleich — zwei Vektoren unterschiedlicher Länge geben hier keinen Fehler, sondern eine falsche
Zahl. Der letzte Abschnitt dieses Notebooks kommt darauf zurück.

</details>

📖 Jetzt auf echten Text. Drei kurze Sätze aus dem Security Operations Center: Die ersten beiden
sagen dasselbe mit anderen Wörtern, der dritte handelt von etwas anderem.

In [ ]:
# ▶️ Drei Sätze, drei Vektoren
SAETZE = [
    "Ein Angreifer kann die Anmeldung umgehen.",             # A
    "Die Authentifizierung lässt sich aushebeln.",           # B — dieselbe Aussage, andere Wörter
    "Die Protokolle liegen 90 Tage im schnellen Zugriff.",   # C — anderes Thema
]

A, B, C = helfer.embed(SAETZE)


def inhaltswoerter(satz):
    """Alle Wörter ab vier Zeichen, klein geschrieben und ohne Satzzeichen."""
    geputzt = (w.strip(".,:;?!") .lower() for w in satz.split())
    return {w for w in geputzt if len(w) > 3}


print(f"{'Paar':<8}{'euklidisch':>12}{'Skalarprod.':>13}{'Kosinus':>10}   gemeinsame Wörter")
print("-" * 66)
for name, i, j, u, v in [("A ↔ B", 0, 1, A, B), ("A ↔ C", 0, 2, A, C), ("B ↔ C", 1, 2, B, C)]:
    gemeinsam = inhaltswoerter(SAETZE[i]) & inhaltswoerter(SAETZE[j])
    print(f"{name:<8}{euklidische_distanz(u, v):>12.4f}{skalarprodukt(u, v):>13.4f}"
          f"{kosinus_aehnlichkeit(u, v):>10.4f}   {', '.join(sorted(gemeinsam)) or 'keine'}")

📖 A und B haben kein einziges Wort ab vier Zeichen gemeinsam. Trotzdem stehen sie sich näher
als jedes andere Paar: der höchste Kosinus, die kleinste Distanz. Eine Suche über
Wortübereinstimmung würde die beiden für unverwandt halten.

Auffällig ist noch etwas: Alle drei Spalten sind sich einig. Dasselbe Paar gewinnt bei jedem der
drei Maße. Das ist kein Zufall — der nächste Abschnitt zeigt, warum.

---
## 3 · Die Maße auf zwei echten Dokumenten

📖 Zwei Dokumente der Wissensbasis, die inhaltlich zusammengehören: das Advisory zu
CVE-2026-3224 und das Post-Mortem des Vorfalls, der auf dieser Schwachstelle beruht.

▶️ Die nächste Zelle bettet die **vollständigen Dokumente** ein — nicht die Chunks — und rechnet
alle drei Maße aus.

In [ ]:
# ▶️ Advisory und Post-Mortem, mit allen drei Maßen
dokumente = helfer.lade_dokumente()
volltext = {d["id"]: d["text"] for d in dokumente}

D1, D2 = "cve-2026-3224", "postmortem-2026-03-14-vpn"
v1, v2 = helfer.embed([volltext[D1], volltext[D2]])

for kennung, v in [(D1, v1), (D2, v2)]:
    print(f"{kennung:<28}{helfer.zaehle_tokens(volltext[kennung]):>6} Tokens   "
          f"Vektorlänge {math.sqrt(skalarprodukt(v, v)):.4f}")

print()
print(f"euklidische Distanz   {euklidische_distanz(v1, v2):>8.4f}   klein = ähnlich")
print(f"Skalarprodukt         {skalarprodukt(v1, v2):>8.4f}   groß  = ähnlich")
print(f"Kosinus-Ähnlichkeit   {kosinus_aehnlichkeit(v1, v2):>8.4f}   groß  = ähnlich")

In [ ]:
# ▶️ Mehrere Dokumentpaare nebeneinander, sortiert nach Kosinus
IDS = ["cve-2026-3224", "postmortem-2026-03-14-vpn", "runbook-patch-management",
       "policy-logging-und-aufbewahrung", "cve-2026-1187"]

dok_vektor = dict(zip(IDS, helfer.embed([volltext[i] for i in IDS])))

zeilen = [(a, b,
           euklidische_distanz(dok_vektor[a], dok_vektor[b]),
           skalarprodukt(dok_vektor[a], dok_vektor[b]),
           kosinus_aehnlichkeit(dok_vektor[a], dok_vektor[b]))
          for i, a in enumerate(IDS) for b in IDS[i + 1:]]
zeilen.sort(key=lambda z: -z[4])

print(f"{'Paar':<58}{'euklidisch':>12}{'Skalarprod.':>13}{'Kosinus':>10}")
print("-" * 93)
for a, b, e, s, k in zeilen:
    print(f"{a[:26] + ' ↔ ' + b[:26]:<58}{e:>12.4f}{s:>13.4f}{k:>10.4f}")

📖 Die Tabelle ist nach Kosinus sortiert. Dabei läuft die Spalte der euklidischen Distanzen
streng aufsteigend und die des Skalarprodukts streng absteigend: Alle drei Maße erzeugen hier
**dieselbe Rangfolge**.

Der Grund steht eine Zelle weiter oben — alle Vektoren haben die Länge 1. Für normalisierte
Vektoren gilt

```
Skalarprodukt(a, b) = Kosinus(a, b)
euklidische Distanz(a, b) = sqrt(2 - 2 · Kosinus(a, b))
```

Das Skalarprodukt **ist** dann der Kosinus, und die Distanz ist eine fallende Funktion davon.
Solange das Modell normalisierte Vektoren liefert, ändert die Wahl des Maßes die Reihenfolge der
Treffer nicht.

▶️ Die nächste Zelle prüft die zweite Gleichung nach und zeigt danach, was passiert, wenn ein
Vektor **nicht** normalisiert ist.

In [ ]:
# ▶️ Die Gleichung nachrechnen, und was ohne Normalisierung passiert
k = kosinus_aehnlichkeit(v1, v2)
print("Bei normalisierten Vektoren:")
print(f"  gemessene Distanz       {euklidische_distanz(v1, v2):.6f}")
print(f"  sqrt(2 - 2 · Kosinus)   {math.sqrt(2 - 2 * k):.6f}")

# derselbe Vektor, nur dreimal so lang — die Richtung bleibt, der Inhalt auch
v2_lang = [3 * x for x in v2]

print()
print(f"{'':<22}{'euklidisch':>12}{'Skalarprod.':>13}{'Kosinus':>10}")
print("-" * 57)
print(f"{'v1 gegen v2':<22}{euklidische_distanz(v1, v2):>12.4f}"
      f"{skalarprodukt(v1, v2):>13.4f}{kosinus_aehnlichkeit(v1, v2):>10.4f}")
print(f"{'v1 gegen 3 · v2':<22}{euklidische_distanz(v1, v2_lang):>12.4f}"
      f"{skalarprodukt(v1, v2_lang):>13.4f}{kosinus_aehnlichkeit(v1, v2_lang):>10.4f}")

📖 Die Richtung ist dieselbe geblieben, nur die Länge des Vektors hat sich verdreifacht. Die
euklidische Distanz wächst, das Skalarprodukt verdreifacht sich — die Kosinus-Ähnlichkeit steht
unverändert da.

Das ist der Grund, warum Kosinus das übliche Maß für Text-Embeddings ist:

* Sie misst die **Richtung** und ignoriert die Länge. Die Bedeutung steckt in der Richtung.
* Nicht jedes Modell normalisiert. Wo die Vektorlänge an der Textlänge hängt, bekäme ein langer
  Chunk allein wegen seiner Länge ein größeres Skalarprodukt als ein kurzer, der besser passt.
* Der Wertebereich ist fest: -1 bis 1. Damit lässt sich ein Schwellenwert festlegen, der nicht
  von der Sammlung abhängt.

Chroma bekommt das Maß beim Anlegen der Collection gesagt: `metadata={"hnsw:space": "cosine"}`.
Zurück gibt Chroma allerdings eine **Distanz**, nämlich `1 - Kosinus`. Deshalb rechnet man dort
`score = 1 - distanz`, um wieder die Ähnlichkeit zu bekommen.

---
## 4 · Die Suche von Hand

📖 Damit ist die Suche vollständig beschrieben:

1. Alle Chunks einbetten — einmal, beim Aufbau der Sammlung.
2. Die Frage einbetten — bei jeder Anfrage.
3. Die Frage gegen **jeden** Chunk vergleichen.
4. Nach Ähnlichkeit sortieren, die besten *n* zurückgeben.

Schritt 3 ist die lineare Suche, auch Brute Force genannt: kein Index, keine Abkürzung, jeder
Chunk wird angefasst.

### 🛠️ Challenge 2: Lineare Suche

Schreibe `suche_linear(frage, chunks, n=5, modell=EMBEDDING_MODELL)`. Rückgabe ist eine
**Liste der besten n Chunks**, absteigend sortiert. Jeder Treffer ist der Chunk selbst, ergänzt
um den Schlüssel `score` mit der Kosinus-Ähnlichkeit zur Frage.

1. Die Chunk-Texte einbetten: `helfer.embed(texte, modell=modell)`.
2. Die Frage einbetten: `helfer.embed([frage], modell=modell)[0]`.
3. Für jeden Chunk `kosinus_aehnlichkeit(frage_vektor, chunk_vektor)` ausrechnen.
4. Absteigend sortieren, die ersten `n` zurückgeben.

*Tipp: `{**chunk, "score": wert}` erzeugt eine Kopie des Chunk-Dicts mit einem zusätzlichen
Feld. Sortiert wird mit `liste.sort(key=lambda t: t["score"], reverse=True)`.*

In [ ]:
def suche_linear(frage, chunks, n=5, modell=EMBEDDING_MODELL):
    """Vergleicht die Frage mit jedem Chunk und gibt die n ähnlichsten zurück."""
    # TODO 1: alle Chunk-Texte und die Frage einbetten
    chunk_vektoren = ...
    frage_vektor = ...

    # TODO 2: je Chunk eine Kopie mit dem zusätzlichen Schlüssel "score" bauen
    bewertet = ...

    # TODO 3: absteigend nach score sortieren und die ersten n zurückgeben
    raise NotImplementedError("Challenge 2: suche_linear() implementieren")


In [ ]:
# ✅ Selbsttest
PROBEFRAGE = "Wie schnell muss ein Notfall-Patch eingespielt werden?"
treffer = suche_linear(PROBEFRAGE, chunks, n=5)

assert len(treffer) == 5, f"Fünf Treffer erwartet, {len(treffer)} bekommen"
assert all("score" in t for t in treffer), "Jeder Treffer braucht einen score"
assert set(chunks[0]) <= set(treffer[0]), "Die Felder des Chunks müssen erhalten bleiben"
assert treffer == sorted(treffer, key=lambda t: -t["score"]), "Absteigend sortieren"
assert all(-1 <= t["score"] <= 1 for t in treffer), "Kosinus liegt zwischen -1 und 1"
assert len({t["chunk_id"] for t in treffer}) == 5, "Kein Chunk darf doppelt vorkommen"
assert len(suche_linear(PROBEFRAGE, chunks, n=1)) == 1, "n muss die Trefferzahl steuern"
assert treffer[0]["score"] > 0.7, "Der beste Treffer sollte über 0,7 liegen"

erwartet = {"runbook-patch-management", "postmortem-2026-03-14-vpn"}
assert erwartet & {t["dok_id"] for t in treffer}, "Patch-Management oder Post-Mortem erwartet"

print("✅ Challenge 2 gelöst")
helfer.zeige_treffer(treffer)

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def suche_linear(frage, chunks, n=5, modell=EMBEDDING_MODELL):
    """Vergleicht die Frage mit jedem Chunk und gibt die n ähnlichsten zurück."""
    chunk_vektoren = helfer.embed([c["text"] for c in chunks], modell=modell)
    frage_vektor = helfer.embed([frage], modell=modell)[0]

    bewertet = [{**chunk, "score": kosinus_aehnlichkeit(frage_vektor, vektor)}
                for chunk, vektor in zip(chunks, chunk_vektoren)]

    bewertet.sort(key=lambda t: t["score"], reverse=True)
    return bewertet[:n]
```

`helfer.embed()` holt die Chunk-Vektoren aus dem Cache, deshalb steht der Aufruf hier ohne
Bedenken in der Suchfunktion. In einem echten System werden die Chunk-Vektoren einmal berechnet
und in der Datenbank abgelegt — bei einer Anfrage wird nur noch die Frage eingebettet.

</details>

In [ ]:
# ▶️ Zwei Beispielfragen
for frage in ["Wie lange werden Firewall-Logs aufbewahrt?",
              "Wie viele Ereignisse pro Sekunde verarbeitet SentinelGrid im Normalbetrieb?"]:
    print(f"❓ {frage}")
    helfer.zeige_treffer(suche_linear(frage, chunks, n=3))
    print()

---
## 5 · Das Skalierungsproblem

📖 Die lineare Suche vergleicht die Frage mit jedem Chunk. Bei 90 Chunks sind das 90 Vergleiche
über je 768 Dimensionen. Bei 1.000.000 Chunks sind es 1.000.000 Vergleiche — und zwar je
Anfrage.

Der Aufwand wächst **linear** mit der Sammlung: doppelt so viele Chunks, doppelt so lange. Und
die Sammlung wächst immer.

▶️ Die nächste Zelle misst die Suche über die 90 Chunks und rechnet hoch.

In [ ]:
# ▶️ Laufzeit messen und hochrechnen
FRAGE = "Wie schnell muss ein Notfall-Patch eingespielt werden?"
chunk_vektoren = helfer.embed([c["text"] for c in chunks])
frage_vektor = helfer.embed([FRAGE])[0]
dimension = len(frage_vektor)

t0 = time.perf_counter()
for _ in range(10):
    werte = [kosinus_aehnlichkeit(frage_vektor, v) for v in chunk_vektoren]
dauer = (time.perf_counter() - t0) / 10
je_chunk = dauer / len(chunks)

print(f"{len(chunks)} Chunks à {dimension} Dimensionen")
print(f"eine Anfrage: {dauer * 1000:.2f} ms   →   je Chunk {je_chunk * 1e6:.1f} µs")
print()
print(f"{'Sammlung':>12}{'Vergleiche':>14}{'Multiplikationen':>20}{'Dauer je Anfrage':>20}")
print("-" * 66)
for groesse in [90, 1_000, 100_000, 1_000_000]:
    print(f"{groesse:>12,}{groesse:>14,}{groesse * dimension:>20,}"
          f"{groesse * je_chunk:>17.2f} s".replace(",", "."))

In [ ]:
# ▶️ Wie die Dauer mit der Sammlungsgröße wächst
GROESSEN = [90, 900, 9_000]
gemessen = []

for groesse in GROESSEN:
    # die Sammlung künstlich vergrößern, indem dieselben Vektoren mehrfach vorkommen
    viele = (chunk_vektoren * (groesse // len(chunk_vektoren) + 1))[:groesse]
    t0 = time.perf_counter()
    werte = [kosinus_aehnlichkeit(frage_vektor, v) for v in viele]
    beste = sorted(range(len(werte)), key=lambda i: -werte[i])[:5]
    gemessen.append(time.perf_counter() - t0)

x = np.array([90, 1e3, 1e4, 1e5, 1e6])

plt.figure(figsize=(8, 4.5))
plt.loglog(x, x * je_chunk, color=ORANGE, linestyle="--", label="hochgerechnet")
plt.loglog(GROESSEN, gemessen, "o", color=BLAU, markersize=8, label="gemessen")
plt.axvline(1e6, color=GRAU, linestyle=":")
plt.text(9e5, 1.5 * min(gemessen), "1.000.000 Chunks ", color=GRAU, va="bottom", ha="right")
plt.xlabel("Chunks in der Sammlung")
plt.ylabel("Dauer einer Anfrage in Sekunden")
plt.title("Lineare Suche: zehnfache Sammlung, zehnfache Dauer")
plt.legend()
plt.show()

for groesse, sekunden in zip(GROESSEN, gemessen):
    print(f"{groesse:>7,}".replace(",", ".") + f" Chunks, Python-Schleife: {sekunden * 1000:>9.1f} ms")

# dieselbe Rechnung als eine einzige Matrixmultiplikation
matrix = np.repeat(np.asarray(chunk_vektoren, dtype=np.float32), 100, axis=0)
q = np.asarray(frage_vektor, dtype=np.float32)
t0 = time.perf_counter()
for _ in range(20):
    aehnlichkeiten = matrix @ q
    beste = np.argpartition(-aehnlichkeiten, 5)[:5]
numpy_dauer = (time.perf_counter() - t0) / 20

print(f"{len(matrix):>7,}".replace(",", ".") + f" Chunks, numpy:            {numpy_dauer * 1000:>9.1f} ms")

📖 Die drei gemessenen Punkte liegen auf der hochgerechneten Geraden. Im doppelt
logarithmischen Diagramm ist sie eine Gerade mit Steigung 1 — genau das ist lineares Wachstum:
Die zehnfache Sammlung kostet die zehnfache Zeit.

Die Implementierung ändert daran nichts. `numpy` rechnet dieselben Vergleiche als eine einzige
Matrixmultiplikation und ist dabei um Größenordnungen schneller — die Gerade rutscht nach
unten, ihre Steigung bleibt.

Genau dagegen gibt es einen Index.

▶️ Dieselbe Suche über Chroma.

In [ ]:
# ▶️ Dieselbe Suche über den Index von Chroma
t0 = time.perf_counter()
for _ in range(20):
    roh = sammlung.query(query_embeddings=[frage_vektor], n_results=5)
chroma_dauer = (time.perf_counter() - t0) / 20

print(f"Python-Schleife über {len(chunks)} Chunks:  {dauer * 1000:>7.2f} ms")
print(f"Chroma über dieselben {len(chunks)} Chunks: {chroma_dauer * 1000:>7.2f} ms")
print()
print("Chroma liefert Distanzen, keine Ähnlichkeiten:")
for chunk_id, abstand in zip(roh["ids"][0], roh["distances"][0]):
    print(f"  {chunk_id:<32} Distanz {abstand:.4f}   →   score {1 - abstand:.4f}")

📖 Bei 90 Chunks gewinnt der Index nichts. Sein Verwaltungsaufwand ist größer als die Ersparnis,
und beide Verfahren liegen in derselben Größenordnung. Ein Index lohnt sich erst, wenn die
Sammlung groß ist — dann allerdings deutlich.

Der Index, den Chroma dafür benutzt, heißt **HNSW**, Hierarchical Navigable Small World. Die
Idee in drei Sätzen: Die Vektoren werden zu einem Graphen verknüpft, in dem jeder Punkt einige
nahe und wenige weit entfernte Nachbarn hat. Eine Suche startet an einem beliebigen Punkt und
geht immer zu dem Nachbarn weiter, der der Frage näher liegt, bis es keinen näheren mehr gibt —
berührt werden nur die Punkte auf dem Weg, nicht alle. Mehrere Ebenen mit immer gröberen
Verknüpfungen halten diesen Weg kurz, sodass der Aufwand nicht linear, sondern etwa
logarithmisch mit der Sammlung wächst.

Der Preis: HNSW ist **approximativ**. Es kann einen Nachbarn übersehen, den die lineare Suche
gefunden hätte. Das Bonus-Notebook `bonus_hnsw` baut den Index selbst und misst, wie oft das
passiert.

---
## 6 · Suche über die Vector Database

📖 Die Collection `wissensbasis` enthält zu jedem Chunk den Vektor, den Text und die Metadaten
`dok_id`, `titel`, `quelle` und `position`. Abgefragt wird sie mit `sammlung.query(...)`:

| Parameter | Bedeutung |
|---|---|
| `query_embeddings` | eine **Liste** von Frage-Vektoren |
| `n_results` | wie viele Treffer je Frage |
| `where` | Filter auf die Metadaten, etwa `{"dok_id": "cve-2026-4410"}` |

Das Ergebnis ist ein Dict aus Listen: `roh["ids"][0]`, `roh["documents"][0]`,
`roh["metadatas"][0]` und `roh["distances"][0]`. Die äußere Liste hat einen Eintrag je Frage —
bei einer einzelnen Frage ist es immer die `0`.

Der Filter ist mehr als Bequemlichkeit. Er wirkt **vor** der Ähnlichkeitssuche: „nur Runbooks",
„nur dieses eine Dokument", „nur Chunks ab Position 3". Damit lassen sich Berechtigungen und
Gültigkeitsbereiche durchsetzen, von denen ein Embedding nichts weiß.

### 🛠️ Challenge 3: Ein Chroma-Ergebnis lesbar machen

Chroma liefert parallele Listen für IDs, Texte, Metadaten und Distanzen. Deine Aufgabe ist,
daraus wieder eine Trefferliste zu bauen:

1. Frage einbetten und sammlung.query() aufrufen.
2. Mit zip() gleichzeitig über die vier Ergebnislisten laufen.
3. Je Position ein Dict mit chunk_id, dok_id, titel, text und score erzeugen.

Chroma verwendet Cosine Distance. Deshalb gilt score = 1 - abstand. Den optionalen Filter
reichst du unverändert an where weiter. Die erste Frage steht jeweils unter Index 0.


In [ ]:
def suche_chroma(frage, n=5, filter=None):
    """Fragt die Collection ab und gibt die n ähnlichsten Chunks als Trefferliste zurück."""
    # TODO 1: die Frage einbetten und die Collection abfragen
    roh = sammlung.query(
        query_embeddings=...,
        n_results=...,
        where=filter,
    )

    # TODO 2: aus ids, documents, metadatas und distances je einen Treffer bauen
    treffer = []
    raise NotImplementedError("Challenge 3: suche_chroma() implementieren")


In [ ]:
# ✅ Selbsttest
FELDER = {"chunk_id", "dok_id", "titel", "text", "score"}
treffer = suche_chroma(PROBEFRAGE, n=5)

assert len(treffer) == 5, f"Fünf Treffer erwartet, {len(treffer)} bekommen"
assert all(set(t) == FELDER for t in treffer), f"Jeder Treffer braucht genau {sorted(FELDER)}"
assert treffer == sorted(treffer, key=lambda t: -t["score"]), "Chroma liefert schon sortiert"
assert all(0 <= t["score"] <= 1 for t in treffer), "score = 1 - Distanz, hier zwischen 0 und 1"
assert treffer[0]["score"] > 0.7, "Der beste Treffer sollte über 0,7 liegen"

RUNBOOKS = ["runbook-incident-response", "runbook-patch-management", "runbook-zugriffskontrolle"]
nur_runbooks = suche_chroma(PROBEFRAGE, n=5, filter={"dok_id": {"$in": RUNBOOKS}})
assert len(nur_runbooks) == 5, "Auch gefiltert kommen fünf Treffer"
assert all(t["dok_id"] in RUNBOOKS for t in nur_runbooks), "Der Filter greift nicht"

ein_dokument = suche_chroma("Welcher Workaround hilft?", n=3, filter={"dok_id": "cve-2026-4410"})
assert all(t["dok_id"] == "cve-2026-4410" for t in ein_dokument), "Der Filter greift nicht"

# gegen die eigene lineare Suche: dieselbe Rechnung, dasselbe Ergebnis
linear = suche_linear(PROBEFRAGE, chunks, n=5)
assert treffer[0]["chunk_id"] == linear[0]["chunk_id"], "Der beste Treffer sollte übereinstimmen"
assert abs(treffer[0]["score"] - linear[0]["score"]) < 1e-3, "Auch der score sollte übereinstimmen"

print("✅ Challenge 3 gelöst")
helfer.zeige_treffer(treffer)

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def suche_chroma(frage, n=5, filter=None):
    """Fragt die Collection ab und gibt die n ähnlichsten Chunks als Trefferliste zurück."""
    roh = sammlung.query(
        query_embeddings=helfer.embed([frage]),
        n_results=n,
        where=filter,
    )

    treffer = []
    for chunk_id, text, metadaten, abstand in zip(
            roh["ids"][0], roh["documents"][0], roh["metadatas"][0], roh["distances"][0]):
        treffer.append({
            "chunk_id": chunk_id,
            "dok_id": metadaten["dok_id"],
            "titel": metadaten["titel"],
            "text": text,
            "score": 1 - abstand,
        })
    return treffer
```

`helfer.embed([frage])` gibt bereits eine Liste mit einem Vektor zurück — genau das, was
`query_embeddings` erwartet.

</details>

In [ ]:
# ▶️ Derselbe Suchtext, drei Filter
SUCHTEXT = "Wie wird ein Zugang gesperrt?"

for beschriftung, filter in [
        ("ohne Filter", None),
        ("nur Runbooks", {"dok_id": {"$in": RUNBOOKS}}),
        ("nur die Zugriffskontrolle", {"dok_id": "runbook-zugriffskontrolle"})]:
    print(f"❓ {SUCHTEXT}   ({beschriftung})")
    helfer.zeige_treffer(suche_chroma(SUCHTEXT, n=3, filter=filter))
    print()

---
## 7 · Messen: Recall@k

📖 Ob eine Suche gut ist, entscheidet nicht der Eindruck, sondern eine Messung über Fragen mit
bekannter Antwort. `daten/fragen.json` enthält zehn davon, jede mit den Dokumenten, in denen die
Antwort steht.

**Recall@k** ist der Anteil der Fragen, bei denen unter den ersten *k* Treffern mindestens ein
erwartetes Dokument steht. Zwei Werte sind interessant:

* **Recall@5** — reicht das, was in den Prompt kommt, um die Frage zu beantworten? Drei bis fünf
  Chunks sind die übliche Größe eines RAG-Prompts.
* **Recall@1** — steht das Richtige ganz oben? Diese Zahl ist strenger und sagt mehr über die
  Rangfolge aus.

### 🛠️ Challenge 4: Recall@k

Schreibe `recall_at_k(fragen, suchfunktion, k=5)`. Für jede Frage:

1. `suchfunktion(frage["frage"], k)` aufrufen — die Funktion gibt eine Trefferliste zurück.
2. Die `dok_id` aller Treffer einsammeln.
3. Die Frage zählt als getroffen, wenn mindestens eine davon in `frage["erwartete_dok_ids"]` steht.

Rückgabe ist der Anteil der getroffenen Fragen, eine Zahl zwischen 0 und 1.

*Tipp: Zwei Mengen haben mit `menge_a & menge_b` einen Durchschnitt, und eine leere Menge ist in
einer `if`-Abfrage `False`.*

In [ ]:
def recall_at_k(fragen, suchfunktion, k=5):
    """Anteil der Fragen, bei denen unter den ersten k Treffern ein erwartetes Dokument steht."""
    getroffen = 0
    for frage in fragen:
        # TODO 1: suchen und die dok_id der Treffer als Menge einsammeln
        gefunden = ...
        # TODO 2: hochzählen, wenn ein erwartetes Dokument dabei ist
        ...
    # TODO 3: den Anteil zurückgeben
    raise NotImplementedError("Challenge 4: recall_at_k() implementieren")


In [ ]:
# ✅ Selbsttest
def _immer_richtig(frage, k):
    return [{"dok_id": d} for d in fragen[0]["erwartete_dok_ids"]]


def _immer_falsch(frage, k):
    return [{"dok_id": "gibt-es-nicht"}] * k


assert recall_at_k(fragen[:1], _immer_richtig, k=5) == 1.0, "Alle Fragen getroffen ergibt 1.0"
assert recall_at_k(fragen, _immer_falsch, k=5) == 0.0, "Keine Frage getroffen ergibt 0.0"
assert recall_at_k([fragen[0], fragen[2]], _immer_richtig, k=5) == 0.5, "Eine von zwei ergibt 0.5"

r5 = recall_at_k(fragen, lambda f, k: suche_chroma(f, n=k), k=5)
r1 = recall_at_k(fragen, lambda f, k: suche_chroma(f, n=k), k=1)

assert 0.0 <= r1 <= r5 <= 1.0, "Recall@1 kann nicht größer sein als Recall@5"
assert r5 >= 0.7, f"Recall@5 sollte über 0,7 liegen, gemessen wurden {r5:.2f}"

print("✅ Challenge 4 gelöst")
print(f"Semantic Search über {len(chunks)} Chunks und {len(fragen)} Fragen:")
print(f"  Recall@1 = {r1:.0%}")
print(f"  Recall@5 = {r5:.0%}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def recall_at_k(fragen, suchfunktion, k=5):
    """Anteil der Fragen, bei denen unter den ersten k Treffern ein erwartetes Dokument steht."""
    getroffen = 0
    for frage in fragen:
        gefunden = {t["dok_id"] for t in suchfunktion(frage["frage"], k)}
        if gefunden & set(frage["erwartete_dok_ids"]):
            getroffen += 1
    return getroffen / len(fragen)
```

Die Suchfunktion wird übergeben, statt fest eingebaut zu sein. Damit misst dieselbe Funktion
jedes Verfahren — nötig ist nur eine Trefferliste mit `dok_id`.

</details>

In [ ]:
# ▶️ Frage für Frage: auf welchem Rang steht das erwartete Dokument?
def rang_des_treffers(frage):
    """Position des ersten erwarteten Dokuments in der vollständigen Rangfolge."""
    alle = suche_linear(frage["frage"], chunks, n=len(chunks))
    for rang, treffer in enumerate(alle, start=1):
        if treffer["dok_id"] in frage["erwartete_dok_ids"]:
            return rang, alle[0]
    return len(alle), alle[0]


print(f"{'Rang':>5}  {'Frage':<58}{'bester Treffer':<33}{'score':>7}")
print("-" * 106)
for frage in fragen:
    rang, bester = rang_des_treffers(frage)
    marke = "✔" if rang == 1 else " "
    print(f"{rang:>4}{marke}  {frage['frage'][:56]:<58}"
          f"{bester['dok_id']:<33}{bester['score']:>7.3f}")

📖 Die Zahlen ordnen sich so ein.

**Recall@5 ist hoch.** Jede der zehn Fragen hat ihr Dokument unter den ersten fünf Treffern. Für
einen RAG-Prompt mit fünf Chunks reicht das: Die Antwort steht im Kontext.

**Recall@1 ist deutlich niedriger.** Bei mehreren Fragen steht das erwartete Dokument auf Rang
zwei oder drei. Semantic Search findet das Themenfeld zuverlässig, die Reihenfolge innerhalb des
Themenfelds ist weniger sicher.

▶️ Zwei Fragen zeigen, was Semantic Search kann. Die nächste Zelle druckt zu jedem Treffer die
Wörter, die er mit der Frage teilt.

In [ ]:
# ▶️ Zwei Fragen und die Wörter, die ihre Treffer mit ihnen teilen
for nummer in [3, 9]:
    frage = fragen[nummer]
    print(f"❓ {frage['frage']}")
    print(f"   erwartet: {frage['erwartete_dok_ids']}")
    for rang, t in enumerate(suche_linear(frage["frage"], chunks, n=3), start=1):
        gemeinsam = inhaltswoerter(frage["frage"]) & inhaltswoerter(t["text"])
        print(f"   {rang}. {t['score']:.3f}  {t['chunk_id']:<36}"
              f"gemeinsame Wörter: {', '.join(sorted(gemeinsam)) or 'keine'}")
        print(f"        {' '.join(t['text'].split())[:110]}…")
    print()

📖 Bei der Frage nach dem letzten Arbeitstag teilt kein einziger der drei Treffer ein Wort ab
vier Zeichen mit der Frage — und trotzdem kommen zwei davon aus dem Runbook Zugriffskontrolle,
das genau diesen Vorgang unter der Überschrift „Austritt" beschreibt. Über
Wortübereinstimmung wäre dieses Dokument nicht zu finden.

Bei der Frage nach der SEV-1-Einstufung landet die Tabelle der Schweregrade aus dem Runbook auf
Rang zwei, hauchdünn hinter einem Post-Mortem-Abschnitt, in dem „SEV-1" nur nebenbei vorkommt.

▶️ Und jetzt der Fall, in dem Semantic Search danebenliegt.

In [ ]:
# ▶️ Exakte Kennungen sind die Schwachstelle
FEHLGRIFF = fragen[8]

print(f"❓ {FEHLGRIFF['frage']}")
print(f"   erwartet: {FEHLGRIFF['erwartete_dok_ids']}")
print()
helfer.zeige_treffer(suche_linear(FEHLGRIFF["frage"], chunks, n=5))

print()
print("Bester score je CVE-Advisory:")
for kennung in ["cve-2026-4410", "cve-2026-3224", "cve-2026-1187", "cve-2025-9042"]:
    nur_dieses = [c for c in chunks if c["dok_id"] == kennung]
    beste = suche_linear(FEHLGRIFF["frage"], nur_dieses, n=1)[0]
    marke = "  ← erwartet" if kennung in FEHLGRIFF["erwartete_dok_ids"] else ""
    print(f"  {kennung:<20}{beste['score']:.4f}{marke}")

📖 Drei der vier Advisories liegen wenige Tausendstel auseinander, und das erwartete ist nicht
das beste. Sie sind sich als Text sehr ähnlich: dieselbe Gliederung, dieselben Fachbegriffe,
dieselben Abschnitte über Bewertung, betroffene Versionen und Workaround. Was sie
unterscheidet, ist eine Nummer — und
`CVE-2026-4410` und `CVE-2026-3224` stehen im Vektorraum dicht beieinander, weil das Modell die
Ziffernfolge nicht als Kennung liest, sondern als Text.

Dasselbe gilt für Versionsnummern, Hostnamen, Ticketnummern und Fehlercodes. Genau dort ist eine
Suche über exakte Wortübereinstimmung im Vorteil.

📖 Noch etwas fällt an den zehn Fragen auf: Sie benutzen die Wörter der Dokumente. „Notfall-Patch",
„Firewall-Logs", „SEV-1" — das steht so in der Wissensbasis. Ein zweiter Fragensatz stellt
dieselben Sachverhalte in anderen Worten.

▶️ Sechs Fragen, deren Wortwahl nicht aus der Wissensbasis stammt.

In [ ]:
# ▶️ Dieselben Sachverhalte, andere Wörter
UMFORMULIERT = [
    ("Wie gefährlich ist die Lücke im VPN-Zugang von NorthPeak?", ["cve-2026-3224"]),
    ("Wie viel Zeit bleibt für ein dringendes Sicherheitsupdate?", ["runbook-patch-management"]),
    ("Wer stuft einen Vorfall als besonders schwerwiegend ein?", ["runbook-incident-response"]),
    ("Wie lange bleiben die Aufzeichnungen der Firewall gespeichert?", ["policy-logging-und-aufbewahrung"]),
    ("Was geschieht mit dem Zugang, wenn jemand die Firma verlässt?", ["runbook-zugriffskontrolle"]),
    ("Wie viel Last hält die SIEM-Plattform im Alltag aus?", ["systemdoku-sentinelgrid"]),
]

UMFORMULIERTE_FRAGEN = [{"frage": f, "erwartete_dok_ids": e} for f, e in UMFORMULIERT]

print("✔ erwartetes Dokument auf Platz 1   ~ unter den ersten fünf   ✗ nicht darunter")
print()
for frage, erwartet in UMFORMULIERT:
    treffer = suche_linear(frage, chunks, n=5)
    dok_ids = [t["dok_id"] for t in treffer]
    marke = "✔" if dok_ids[0] in erwartet else ("~" if set(dok_ids) & set(erwartet) else "✗")
    print(f"{marke} {frage}")
    print(f"    Platz 1: {treffer[0]['chunk_id']:<36}({treffer[0]['score']:.3f})   "
          f"erwartet: {erwartet[0]}")

▶️ Zum Vergleich dieselbe Messung mit einer Suche über Wortübereinstimmung. Verfahren ist
**BM25**: Es zählt, welche Suchwörter im Chunk vorkommen, gewichtet sie danach, wie selten sie
im Bestand sind, und rechnet die Chunklänge heraus. Die nächste Zelle baut sie als
Vergleichsmaßstab und misst beide Verfahren über beide Fragensätze.

In [ ]:
# ▶️ Wortübereinstimmung gegen Vektorraum, über beide Fragensätze
STOPPWOERTER = {
    "der", "die", "das", "des", "dem", "den", "ein", "eine", "einer", "eines", "einem", "einen",
    "und", "oder", "aber", "auch", "als", "wie", "was", "wer", "wo", "wann", "warum", "welche",
    "ist", "sind", "war", "waren", "wird", "werden", "wurde", "wurden", "hat", "haben", "kann",
    "in", "im", "an", "am", "auf", "aus", "bei", "mit", "nach", "von", "vom", "vor", "zu", "zum",
    "zur", "für", "über", "unter", "durch", "gegen", "ohne", "um", "bis", "seit", "je", "pro",
    "ich", "du", "er", "sie", "es", "wir", "ihr", "man", "sich", "dieser", "diese", "dieses",
    "nicht", "kein", "keine", "nur", "noch", "schon", "so", "dass", "wenn", "dann", "sein",
}


def tokenisiere(text):
    """Zerlegt einen Text in kleingeschriebene Suchwörter ohne Stoppwörter."""
    woerter = re.findall(r"[a-z0-9äöüß]+(?:-[a-z0-9äöüß]+)*", text.lower())
    return [w for w in woerter if w not in STOPPWOERTER]


bm25 = BM25Okapi([tokenisiere(c["text"]) for c in chunks], k1=1.5, b=0.75)


def suche_bm25(frage, n=5):
    """Die n Chunks mit dem höchsten BM25-Score."""
    werte = bm25.get_scores(tokenisiere(frage))
    beste = sorted(range(len(chunks)), key=lambda i: -werte[i])[:n]
    return [{**chunks[i], "score": float(werte[i])} for i in beste]


FRAGENSAETZE = [("Fragen aus daten/fragen.json", fragen),
                ("dieselben Sachverhalte, umformuliert", UMFORMULIERTE_FRAGEN)]

print(f"{'Fragensatz':<38}{'Wort @1':>9}{'Wort @5':>9}{'Vektor @1':>11}{'Vektor @5':>11}")
print("-" * 78)
for name, satz in FRAGENSAETZE:
    wort_1 = recall_at_k(satz, lambda f, k: suche_bm25(f, n=k), k=1)
    wort_5 = recall_at_k(satz, lambda f, k: suche_bm25(f, n=k), k=5)
    vektor_1 = recall_at_k(satz, lambda f, k: suche_linear(f, chunks, n=k), k=1)
    vektor_5 = recall_at_k(satz, lambda f, k: suche_linear(f, chunks, n=k), k=5)
    print(f"{name:<38}{wort_1:>9.0%}{wort_5:>9.0%}{vektor_1:>11.0%}{vektor_5:>11.0%}")

📖 Kein Verfahren gewinnt auf ganzer Linie.

**Bei den Fragen aus `fragen.json` liegt die Wortsuche vorn.** Beide finden für alle zehn Fragen
das erwartete Dokument unter den ersten fünf. Auf Platz 1 ist die Wortsuche besser — kein
Wunder: Diese Fragen benutzen die Wörter der Dokumente, und genau dafür ist sie gebaut.

**Bei den umformulierten Fragen dreht sich der Platz 1.** Semantic Search findet dort doppelt so
oft das richtige Dokument ganz oben. Unter den ersten fünf bleibt die Wortsuche trotzdem vorn:
Sie profitiert davon, dass auch umformulierte Fragen noch einzelne wörtliche Anker enthalten —
*Firewall*, *NorthPeak*, *Zugang*.

**Und bei exakten Kennungen verliert Semantic Search**, wie die Frage nach CVE-2026-4410 gezeigt
hat.

Drei Befunde, drei Richtungen. Das ist die Ausgangslage für **Hybrid Search**: beide Verfahren
laufen lassen und ihre Rangfolgen zusammenführen.

---
## 8 · Das Embedding-Modell wechseln

📖 `nomic-embed-text` ist eine Wahl, keine Notwendigkeit. `bge-m3` ist ein anderes
Embedding-Modell: größer, mehrsprachig trainiert, und es gibt 1024 statt 768 Dimensionen zurück.

`helfer.embed()` nimmt den Modellnamen als Parameter, und `suche_linear` reicht ihn durch. Damit
lässt sich derselbe Aufbau mit einem zweiten Modell messen.

### 🛠️ Challenge 5: Zwei Embedding-Modelle fair vergleichen

Baue messe_modell(). Die Funktion liefert Modellname, Vektordimension, Recall@1 und Recall@5.

Der zentrale Lernschritt ist die faire Vergleichsbedingung: Frage und Chunks müssen mit
demselben Embedding-Modell eingebettet werden. Das Gerüst trennt Einbetten, Suchen und Messen.

Tipp: Definiere innen suche(frage, k), die suche_linear() mit dem Modell aufruft. Übergib diese
Funktion anschließend an recall_at_k().


In [ ]:
def messe_modell(modell):
    """Dimension und Recall eines Embedding-Modells über die Evaluationsfragen."""
    # TODO 1: alle Chunk-Texte mit diesem Modell einbetten
    chunk_vektoren = ...

    # TODO 2: eine Suchfunktion bauen, die frage und k nimmt und modell durchreicht

    # TODO 3: das Dict mit den vier Schlüsseln zurückgeben
    raise NotImplementedError("Challenge 5: messe_modell() implementieren")


In [ ]:
# ✅ Selbsttest
messung = messe_modell(EMBEDDING_MODELL)

assert set(messung) == {"modell", "dimension", "recall@1", "recall@5"}, "Die Schlüssel stimmen nicht"
assert messung["modell"] == EMBEDDING_MODELL, "Der Modellname gehört unverändert ins Ergebnis"
assert messung["dimension"] == 768, \
    f"nomic-embed-text hat 768 Dimensionen, gemessen wurden {messung['dimension']}"
assert 0.0 <= messung["recall@1"] <= messung["recall@5"] <= 1.0, \
    "Recall@1 kann nicht größer sein als Recall@5"
assert messung["recall@5"] >= 0.7, "Recall@5 sollte über 0,7 liegen"

print("✅ Challenge 5 gelöst")
print(messung)

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def messe_modell(modell):
    """Dimension und Recall eines Embedding-Modells über die Evaluationsfragen."""
    chunk_vektoren = helfer.embed([c["text"] for c in chunks], modell=modell)

    def suche(frage, k):
        return suche_linear(frage, chunks, n=k, modell=modell)

    return {
        "modell": modell,
        "dimension": len(chunk_vektoren[0]),
        "recall@1": recall_at_k(fragen, suche, k=1),
        "recall@5": recall_at_k(fragen, suche, k=5),
    }
```

Die innere Funktion `suche` schließt `modell` ein. Ohne diesen Umweg würde `recall_at_k` immer
mit dem Standardmodell suchen, und die Messung wäre für beide Modelle dieselbe.

</details>

▶️ Die nächste Zelle misst beide Modelle. Sie misst zusätzlich, wie lange der Server für ein
einzelnes Embedding braucht — direkt über den Client, damit der Cache das Ergebnis nicht
verfälscht.

Beim ersten Durchlauf muss `bge-m3` alle 90 Chunks einbetten; danach stehen die Vektoren im
Cache.

In [ ]:
# ▶️ Beide Modelle nebeneinander
def antwortzeit(modell, text):
    """Dauer eines einzelnen Embedding-Aufrufs am Server, am Cache vorbei."""
    client.embeddings.create(model=modell, input=[text])   # aufwärmen: Ollama lädt das Modell
    zeiten = []
    for _ in range(3):
        t0 = time.perf_counter()
        client.embeddings.create(model=modell, input=[text])
        zeiten.append(time.perf_counter() - t0)
    return min(zeiten)


messungen = [messe_modell("nomic-embed-text"), messe_modell("bge-m3")]

print(f"{'Modell':<20}{'Dimension':>11}{'Recall@1':>10}{'Recall@5':>10}{'ein Embedding':>16}")
print("-" * 67)
for m in messungen:
    zeit = antwortzeit(m["modell"], FRAGE)
    print(f"{m['modell']:<20}{m['dimension']:>11}{m['recall@1']:>9.0%}"
          f"{m['recall@5']:>10.0%}{zeit * 1000:>13.0f} ms")

In [ ]:
# ▶️ Dieselbe Frage, beide Modelle
print(f"❓ {FEHLGRIFF['frage']}")
print(f"   erwartet: {FEHLGRIFF['erwartete_dok_ids']}")
for modell in ["nomic-embed-text", "bge-m3"]:
    print()
    print(f"— {modell} —")
    helfer.zeige_treffer(suche_linear(FEHLGRIFF["frage"], chunks, n=3, modell=modell))

📖 Der Wechsel bringt beim Recall@1 einen Schritt und beim Recall@5 nichts — beide Modelle
haben schon alle zehn Fragen unter den ersten fünf Treffern. Bezahlt wird der Schritt mit
Rechenzeit: Ein Embedding dauert bei `bge-m3` ein Vielfaches, und das gilt für jeden Chunk beim
Aufbau **und** für jede Frage im Betrieb.

Die beiden Modelle sortieren außerdem unterschiedlich, und ihre scores liegen auf
unterschiedlichen Niveaus. Ein Schwellenwert, der für das eine Modell passt, passt für das
andere nicht.

Drei Punkte gehören zu jedem Modellwechsel:

**Embeddings verschiedener Modelle sind nicht vergleichbar.** Sie haben unterschiedliche
Dimensionen, und selbst bei gleicher Dimension bedeuten die Achsen etwas anderes. Ein Vektor aus
Modell A und ein Vektor aus Modell B haben keine gemeinsame Grundlage.

**Ein Modellwechsel berechnet die ganze Sammlung neu.** Jeder Chunk muss durch das neue Modell.
Bei 90 Chunks ist das ein Moment, bei einer Million ist es ein Projekt — mit Kosten, Laufzeit
und einer zweiten Collection, solange beide parallel gebraucht werden.

**Frage und Chunks müssen durch dasselbe Modell.** Sonst liegt die Frage in einem anderen Raum
als die Chunks, und die Treffer sind beliebig.

▶️ Die letzte Zelle zeigt, warum der dritte Punkt gefährlich ist: Der Fehler meldet sich nicht.

In [ ]:
# ▶️ Derselbe Satz durch zwei Modelle
a = helfer.embed([BEISPIEL], modell="nomic-embed-text")[0]
b = helfer.embed([BEISPIEL], modell="bge-m3")[0]

print(f"Text: {BEISPIEL!r}")
print(f"  nomic-embed-text: {len(a)} Dimensionen")
print(f"  bge-m3:           {len(b)} Dimensionen")
print()
print("Kosinus-Ähnlichkeit über die ersten 768 Dimensionen, zwei Modelle:")
print(f"  {kosinus_aehnlichkeit(a, b):+.4f}")
print("Zum Vergleich, derselbe Text gegen sich selbst im selben Modell:")
print(f"  {kosinus_aehnlichkeit(a, a):+.4f}")

📖 Zwei Vektoren desselben Satzes, und die Ähnlichkeit liegt nahe null. `zip()` schneidet den
längeren Vektor stillschweigend ab, die Rechnung läuft durch und liefert eine Zahl. Die Zahl ist
bedeutungslos.

Eine Vector Database schützt an dieser Stelle zum Teil: Chroma lehnt Vektoren mit falscher
Dimension beim Schreiben ab. Bei gleicher Dimension und unterschiedlichem Modell hilft auch das
nicht mehr — dann bleibt nur, den Modellnamen zur Collection zu notieren.

▶️ Die letzte Zelle legt die Trefferlisten beider Fragensätze in `daten/03_semantic_treffer.json`
ab — dieselbe Form, in der auch andere Verfahren ihre Ergebnisse hinterlegen.

In [ ]:
# ▶️ Die Ergebnisse für spätere Auswertungen sichern
def als_eintrag(frage, erwartete_dok_ids):
    """Frage, erwartete Dokumente und die fünf besten Treffer als Dict."""
    return {
        "frage": frage,
        "erwartete_dok_ids": erwartete_dok_ids,
        "treffer": [{"chunk_id": t["chunk_id"], "score": round(t["score"], 4)}
                    for t in suche_linear(frage, chunks, n=5)],
    }


ergebnis = {
    "verfahren": "cosine",
    "modell": EMBEDDING_MODELL,
    "fragen": [als_eintrag(f["frage"], f["erwartete_dok_ids"]) for f in fragen],
    "umformuliert": [als_eintrag(f, e) for f, e in UMFORMULIERT],
}

ziel = helfer.DATEN / "03_semantic_treffer.json"
ziel.write_text(json.dumps(ergebnis, ensure_ascii=False, indent=1), encoding="utf-8")

print(f"{ziel.name}: {len(ergebnis['fragen'])} Fragen, "
      f"{len(ergebnis['umformuliert'])} umformulierte Fragen")
print(json.dumps(ergebnis["fragen"][0], ensure_ascii=False, indent=1)[:400], "…")

---
## 9 · Was du gebaut hast

* `skalarprodukt()`, `euklidische_distanz()`, `kosinus_aehnlichkeit()` — die drei Maße aus dem
  Vektorraum, von Hand gerechnet und gegen bekannte Werte geprüft.
* `suche_linear()` — die Suche ohne Index: Frage einbetten, gegen jeden Chunk vergleichen,
  sortieren.
* `suche_chroma()` — dieselbe Suche über die Collection `wissensbasis`, mit Metadatenfilter auf
  einzelne Dokumente oder Dokumentarten.
* `recall_at_k()` — die Messung über Fragen mit bekannter Antwort, auf zwei Fragensätzen.
* `messe_modell()` — derselbe Aufbau mit einem zweiten Embedding-Modell, verglichen über
  Dimension, Recall und Antwortzeit.

Die Zahlen zum Mitnehmen. **Über die zehn Fragen aus `fragen.json` liegt Recall@5 bei 100
Prozent, Recall@1 bei 60 Prozent** — das Themenfeld wird zuverlässig gefunden, die Rangfolge
darin nicht. Bei umformulierten Fragen findet Semantic Search doppelt so oft das richtige
Dokument auf Platz 1 wie eine Suche über Wortübereinstimmung; unter den ersten fünf Treffern
bleibt sie hinter ihr. Und bei exakten Kennungen liegt sie daneben: `CVE-2026-4410` und
`CVE-2026-3224` stehen im Vektorraum dicht beieinander, im Betrieb sind es zwei verschiedene
Schwachstellen.

Semantic Search ist also kein Ersatz für die Wortsuche, sondern ihr Gegenstück. Beides zusammen
heißt **Hybrid Search**.

Dazu die Zahl aus dem Skalierungsabschnitt: Bei 1.000.000 Chunks kostet eine lineare Suche eine
Million Vergleiche je Anfrage. Deshalb hat eine Vector Database einen Index.

---
### 🔬 Bonus — ohne Lösung

**1. Präfixe für Frage und Chunk.** `nomic-embed-text` ist mit zwei Präfixen trainiert:
`search_document: ` vor jedem Chunk, `search_query: ` vor jeder Frage. Miss, was sie bringen:

1. Eine Kopie der Chunk-Liste anlegen, in der jeder `text` das Dokument-Präfix trägt.
2. Eine Variante von `suche_linear`, die die Frage mit dem Query-Präfix einbettet.
3. `recall@1` und `recall@5` gegen die Messung ohne Präfix stellen.

Ändert sich Recall@5 überhaupt, oder nur die Rangfolge innerhalb der ersten fünf?

**2. Den Titel mit einbetten.** Ein Chunk aus der Mitte eines Dokuments enthält den Namen des
Dokuments nicht mehr — im Vektor fehlt damit die Zuordnung. Bette `titel + "\n\n" + text` statt
nur `text` ein und miss erneut. Interessant ist besonders die Frage nach dem Workaround für
CVE-2026-4410: Reicht der Titel, um die vier Advisories auseinanderzuhalten?

**3. Ein Schwellenwert für „kein Treffer".** Stelle eine Frage, deren Antwort nicht in der
Wissensbasis steht, etwa nach der Urlaubsregelung. Vergleiche die scores mit denen der zehn
Evaluationsfragen. Ab welchem Wert lässt sich zuverlässig sagen, dass die Sammlung nichts
Passendes enthält — und wie viele richtige Treffer würde dieser Schwellenwert mit wegwerfen?